In [ ]:
from pathlib import Path
import sys
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
import optuna

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits
from src.features.build_features import build_features

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PREDICTIONS = PROJECT_ROOT / "data" / "predictions"
REPORTS = PROJECT_ROOT / "reports"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Загружаем данные и строим фичи
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)

folds = make_time_folds(train["TransactionDT"], n_splits=5)
target = train["isFraud"].astype(np.int8)

train_fe, test_fe = build_features(
    train=train.copy(),
    test=test.copy(),
    target=target,
    folds=folds,
    verbose=False,
)

# Загружаем селектированные фичи
with open(REPORTS / "selected_features_v4.json") as f:
    features_to_keep = json.load(f)["features_to_keep"]

y = train_fe["isFraud"].astype(np.int8)
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "uid"]
X = train_fe.drop(columns=drop_cols)[features_to_keep]

# CatBoost требует явно список категориальных колонок по именам
cat_cols = X.select_dtypes(include=["category"]).columns.tolist()

# Критично для CatBoost: категории должны быть строками или int, НЕ float
# Заполним NaN в категориальных явно строкой "NA"
for col in cat_cols:
    X[col] = X[col].astype("string").fillna("NA").astype("category")

print(f"X: {X.shape}")
print(f"Cat cols: {len(cat_cols)}")
print(f"Memory: {X.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

In [ ]:
# CatBoost дефолтные (разумные для fraud detection) параметры
cb_params = {
    "iterations": 3000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 3,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": 0,
    "early_stopping_rounds": 100,
    "thread_count": -1,
}

splits = expanding_window_splits(folds, n_splits=5)
oof_preds_cb_base = np.zeros(len(X), dtype=np.float32)
cv_scores_cb_base = []
models_cb_base = []

for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
    t0 = time.time()
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    valid_pool = Pool(X_va, y_va, cat_features=cat_cols)

    model = CatBoostClassifier(**cb_params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    preds = model.predict_proba(valid_pool)[:, 1]
    oof_preds_cb_base[valid_idx] = preds
    fold_auc = roc_auc_score(y_va, preds)
    cv_scores_cb_base.append(fold_auc)
    models_cb_base.append(model)

    elapsed = time.time() - t0
    print(f"Fold {fold_num}: AUC = {fold_auc:.5f} "
          f"(best_iter={model.tree_count_}, time={elapsed:.0f}s)")

print()
print(f"CV AUC: {np.mean(cv_scores_cb_base):.5f} ± {np.std(cv_scores_cb_base):.5f}")
print()
print(f"LGBM v5 (tuned): CV 0.93356")
print(f"CatBoost default: CV {np.mean(cv_scores_cb_base):.5f}")

In [ ]:
# Test predictions с CatBoost default
X_test_full = test_fe.drop(columns=["TransactionID", "TransactionDT", "uid"])
X_test = X_test_full[features_to_keep].copy()

# Та же предобработка категорий, что и для train
for col in cat_cols:
    X_test[col] = X_test[col].astype("string").fillna("NA").astype("category")

test_preds_cb = np.zeros(len(X_test), dtype=np.float32)
for model in models_cb_base:
    test_pool = Pool(X_test, cat_features=cat_cols)
    p = model.predict_proba(test_pool)[:, 1]
    test_preds_cb += p / len(models_cb_base)

print(f"CatBoost test predictions: min={test_preds_cb.min():.4f}, "
      f"max={test_preds_cb.max():.4f}, mean={test_preds_cb.mean():.4f}")

# Сохраняем OOF и sub
oof_cb_df = pd.DataFrame({
    "TransactionID": train_fe["TransactionID"],
    "isFraud_true": y,
    "isFraud_pred": oof_preds_cb_base,
    "fold": folds,
})
oof_cb_df.to_csv(DATA_PREDICTIONS / "oof_catboost_default_v5b.csv", index=False)

sub_cb = pd.DataFrame({
    "TransactionID": test_fe["TransactionID"],
    "isFraud": test_preds_cb,
})
sub_cb.to_csv(DATA_PREDICTIONS / "sub_catboost_default_v5b.csv", index=False)
print("Saved: oof + sub for CatBoost default")

In [ ]:
# Загружаем OOF предикты обеих моделей
oof_lgbm = pd.read_csv(DATA_PREDICTIONS / "oof_lgbm_tuned_v5.csv")
oof_cb = pd.read_csv(DATA_PREDICTIONS / "oof_catboost_default_v5b.csv")

# Сортируем одинаково на всякий случай
oof_lgbm = oof_lgbm.sort_values("TransactionID").reset_index(drop=True)
oof_cb = oof_cb.sort_values("TransactionID").reset_index(drop=True)

# Проверка — TransactionID и target должны совпадать
assert (oof_lgbm["TransactionID"] == oof_cb["TransactionID"]).all()
assert (oof_lgbm["isFraud_true"] == oof_cb["isFraud_true"]).all()

y_true = oof_lgbm["isFraud_true"].values
fold_col = oof_lgbm["fold"].values
p_lgbm = oof_lgbm["isFraud_pred"].values
p_cb = oof_cb["isFraud_pred"].values

# Валидные строки (без fold 0 — там нет OOF-предиктов)
valid = fold_col > 0

# 1. Корреляция предиктов — ключевой показатель для blend
from scipy.stats import pearsonr, spearmanr
pearson = pearsonr(p_lgbm[valid], p_cb[valid])[0]
spearman = spearmanr(p_lgbm[valid], p_cb[valid])[0]
print(f"Pearson correlation : {pearson:.4f}")
print(f"Spearman correlation: {spearman:.4f}")
print()

# 2. Индивидуальные скоры
auc_lgbm = roc_auc_score(y_true[valid], p_lgbm[valid])
auc_cb = roc_auc_score(y_true[valid], p_cb[valid])
print(f"LGBM OOF AUC:     {auc_lgbm:.5f}")
print(f"CatBoost OOF AUC: {auc_cb:.5f}")
print()

# 3. Простой average (по вероятностям)
p_avg = 0.5 * p_lgbm + 0.5 * p_cb
auc_avg = roc_auc_score(y_true[valid], p_avg[valid])
print(f"Average blend (50/50):     AUC {auc_avg:.5f} (Δvs LGBM: {auc_avg - auc_lgbm:+.5f})")

# 4. Rank average — робастнее, потому что инвариантен к монотонной калибровке
from scipy.stats import rankdata
r_lgbm = rankdata(p_lgbm)
r_cb = rankdata(p_cb)
p_rank = 0.5 * r_lgbm + 0.5 * r_cb
auc_rank = roc_auc_score(y_true[valid], p_rank[valid])
print(f"Rank average (50/50):      AUC {auc_rank:.5f} (Δvs LGBM: {auc_rank - auc_lgbm:+.5f})")

# 5. Weighted blend — LGBM сильнее, поэтому даём ему больше веса
# Пробуем несколько вариантов веса LGBM
print()
print("Weighted rank blends:")
for w_lgbm in [0.6, 0.7, 0.8, 0.9]:
    p_w = w_lgbm * r_lgbm + (1 - w_lgbm) * r_cb
    auc_w = roc_auc_score(y_true[valid], p_w[valid])
    print(f"  {w_lgbm:.1f} LGBM + {1-w_lgbm:.1f} CB:  AUC {auc_w:.5f} "
          f"(Δvs LGBM: {auc_w - auc_lgbm:+.5f})")